# 03 — Relative Difference Scoring (RDS)

**Goal:** Compute the **Relative Difference Score** for each template — the core innovation of the algorithm.

In the previous notebook we saw that raw reward rates can be misleading because eligibility is confounded with user quality. RDS fixes this by comparing each template against its *actual competition* in every event.

This notebook answers:
- What is a counterfactual baseline?
- How does RDS remove confounding?
- Does the template ranking change after applying RDS?

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample
from src.scoring.difference_score import (
    compute_template_reward_rates,
    compute_counterfactual_baseline,
    compute_relative_difference_scores_fast,
)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
print("Ready!")

---
## 1. Load Data & Recompute Reward Rates

In [ ]:
train_df = load_sample(n_rows=100_000, split="train")
reward_rates, counts = compute_template_reward_rates(train_df)

---
## 2. The Counterfactual Baseline — Concept

For **each notification event** we ask: *"If Duolingo had chosen a template at random from the eligible set, what reward would we expect?"*

$$\text{baseline}(\text{event}) = \frac{1}{|\text{eligible}|} \sum_{t \in \text{eligible}} \text{reward\_rate}(t)$$

This baseline is **different for every event** because eligible sets vary. That's the key: we compare each template against *its actual competition*, not against a fixed global average.

In [ ]:
# Worked example with a single event
example_row = train_df.iloc[0]
eligible = example_row["eligible_templates"]
if isinstance(eligible, str):
    import ast
    eligible = ast.literal_eval(eligible)

print(f"Event 0:")
print(f"  Selected template: {example_row['selected_template']}")
print(f"  Reward (engaged?):  {example_row['session_end_completed']}")
print(f"  Eligible templates: {eligible}")
print()

# Show reward rate of each eligible template
print(f"  Reward rates of eligible templates:")
for t in eligible:
    print(f"    {t}: {reward_rates.get(t, 0):.4f}")

# Compute baseline
bl = compute_counterfactual_baseline(eligible, reward_rates)
print(f"\n  Counterfactual baseline = average of above = {bl:.4f}")

# Compute diff
diff = example_row["session_end_completed"] - bl
print(f"  Diff = reward ({example_row['session_end_completed']}) - baseline ({bl:.4f}) = {diff:+.4f}")
if diff > 0:
    print(f"  → This event performed BETTER than expected under random selection.")
else:
    print(f"  → This event performed WORSE than expected under random selection.")

---
## 3. Computing Relative Difference Scores

The RDS for template *a* is the **average diff** across all events where template *a* was selected:

$$\text{RDS}(a) = \frac{1}{n_a} \sum_{\text{events where } a \text{ was sent}} \Big( \text{reward} - \text{baseline}(\text{event}) \Big)$$

- **Positive RDS** → template consistently outperforms its peers  
- **Negative RDS** → template consistently underperforms  
- **Near zero** → about average

In [ ]:
rds_scores = compute_relative_difference_scores_fast(train_df, reward_rates)

---
## 4. Raw Reward Rate vs RDS — Ranking Comparison

This is the most important chart: does the ranking change when we switch from raw rates to RDS?

In [ ]:
# Build comparison table
templates = sorted(reward_rates.keys())

comparison = pd.DataFrame({
    "template": templates,
    "raw_rate": [reward_rates[t] for t in templates],
    "rds": [rds_scores[t] for t in templates],
    "count": [counts[t] for t in templates],
})

comparison["raw_rank"] = comparison["raw_rate"].rank(ascending=False).astype(int)
comparison["rds_rank"] = comparison["rds"].rank(ascending=False).astype(int)
comparison["rank_change"] = comparison["raw_rank"] - comparison["rds_rank"]

print("Raw Reward Rate vs Relative Difference Score:\n")
print(comparison.sort_values("rds_rank")[
    ["template", "raw_rate", "raw_rank", "rds", "rds_rank", "rank_change", "count"]
].to_string(index=False))

movers = comparison[comparison["rank_change"] != 0]
if len(movers) > 0:
    print(f"\n{len(movers)} template(s) changed rank after applying RDS!")
else:
    print(f"\nNo rank changes — raw rates and RDS agree on this sample.")

In [ ]:
# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Raw reward rates
raw_sorted = comparison.sort_values("raw_rate", ascending=True)
axes[0].barh(raw_sorted["template"], raw_sorted["raw_rate"],
             color="steelblue", edgecolor="white")
axes[0].set_title("Raw Reward Rate", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Rate")
for i, (_, row) in enumerate(raw_sorted.iterrows()):
    axes[0].text(row["raw_rate"] + 0.001, i, f"{row['raw_rate']:.4f}",
                va="center", fontsize=9)

# Right: RDS
rds_sorted = comparison.sort_values("rds", ascending=True)
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in rds_sorted["rds"]]
axes[1].barh(rds_sorted["template"], rds_sorted["rds"],
             color=colors, edgecolor="white")
axes[1].axvline(x=0, color="black", linewidth=0.8)
axes[1].set_title("Relative Difference Score", fontsize=13, fontweight="bold")
axes[1].set_xlabel("RDS")
for i, (_, row) in enumerate(rds_sorted.iterrows()):
    offset = 0.0002 if row["rds"] >= 0 else -0.0002
    ha = "left" if row["rds"] >= 0 else "right"
    axes[1].text(row["rds"] + offset, i, f"{row['rds']:+.5f}",
                va="center", ha=ha, fontsize=9)

plt.suptitle("Raw Rate vs RDS — Does the ranking change?", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Interpreting the Scores

The RDS values are **small numbers** — typically between -0.01 and +0.01. This is expected:

- A template that's 1% better than its peers has RDS ≈ +0.01
- Even a 1-2% lift matters enormously at Duolingo's scale (hundreds of millions of notifications)

The sign tells the story:
- **Positive RDS:** This template genuinely helps users engage, even after adjusting for who sees it
- **Negative RDS:** This template underperforms regardless of user quality
- **Near zero:** Average — not particularly good or bad

In [ ]:
# Distribution of RDS values
fig, ax = plt.subplots(figsize=(8, 4))

rds_values = [rds_scores[t] for t in templates]
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in rds_values]

ax.bar(templates, rds_values, color=colors, edgecolor="white", width=0.6)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_title("RDS by Template (green = outperformer, red = underperformer)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Template")
ax.set_ylabel("Relative Difference Score")

plt.tight_layout()
plt.show()

best = max(rds_scores, key=rds_scores.get)
worst = min(rds_scores, key=rds_scores.get)
print(f"Best template by RDS:  {best} ({rds_scores[best]:+.6f})")
print(f"Worst template by RDS: {worst} ({rds_scores[worst]:+.6f})")
print(f"Spread: {rds_scores[best] - rds_scores[worst]:.6f}")

---
## 6. Summary

| Concept | What it does |
|---------|-------------|
| **Counterfactual baseline** | For each event, average reward rate of all eligible templates |
| **Diff** | Observed reward minus counterfactual baseline |
| **RDS** | Average diff per template — measures genuine quality |

**Key takeaway:** RDS removes the confounding effect of user quality. A template with high raw reward rate but low/negative RDS was benefiting from being shown to active users — it's not actually a good template.

**Problem remaining:** Templates with few observations have noisy RDS values. We need to regularize.

**Next notebook:** `04_bayesian_smoothing_and_recency.ipynb` — smooth the scores with Bayesian shrinkage and add recency penalties.